Inferential Statistics: Welch Two-Sample t-Test

This notebook formally tests whether the mean SoT/90 for Forwards is higher than for Midfielders.

- H₀: μForward = μMidfielder   
- H₁:μForward > μMidfielder
- α = 0.05**

Welch's independent two-sample t-test is used because it does not require the two groups to have equal variances.


In [1]:
#import necessary libraries for data manipulation, statistical analysis, and file handling.
import pandas as pd
import scipy.stats as st # im
import os

# Check if the input file exists before attempting to read it. If the file does not exist, raise a FileNotFoundError with a message indicating that Notebook 2 should be run first to create the stratified sample CSV.
input_file = "sampled_player_data.csv"
if not os.path.exists(input_file):
    raise FileNotFoundError("Run Notebook 2 first to create the stratified sample CSV.")

sample_df = pd.read_csv(input_file) # Read the sample data from the CSV file into a pandas DataFrame named sample_df. This DataFrame will be used for further analysis and visualization of the sampled player data.

sample_fw = sample_df.loc[ # Extract the "SoT/90" values for the "Forward" position group and remove any missing values.
    sample_df["Position_Group"] == "Forward", "SoT/90"
].dropna()

sample_mid = sample_df.loc[ # Extract the "SoT/90" values for the "Midfielder" position group and remove any missing values.
    sample_df["Position_Group"] == "Midfielder", "SoT/90"
].dropna()

# Display the number of observations in each position group after removing missing values. This provides insight into the sample size for each group, which is important for statistical analysis and interpretation of results.
print("Forward n:", len(sample_fw)) 
print("Midfielder n:", len(sample_mid))


Forward n: 64
Midfielder n: 64


In [2]:
#nspect sample normality.
# With n=64 in each group, the t-test is generally fairly robust to moderate non-normality,
# but the distributions and outliers should still be discussed.
fw_shapiro = st.shapiro(sample_fw) # Perform the Shapiro-Wilk test for normality on the "Forward" position group sample. The test checks whether the sample data follows a normal distribution, which is an important assumption for many statistical tests, including the t-test.
mid_shapiro = st.shapiro(sample_mid) # Perform the Shapiro-Wilk test for normality on the "Midfielder" position group sample. Similar to the previous test, this checks whether the sample data for midfielders follows a normal distribution.

# Display the p-values from the Shapiro-Wilk tests for both position groups. The p-value indicates the probability of observing the data if the null hypothesis (that the data is normally distributed) is true. A low p-value (typically < 0.05) suggests that the data may not be normally distributed, which could affect the validity of subsequent statistical tests.
print(f"Forward Shapiro-Wilk p-value: {fw_shapiro.pvalue:.4f}")
print(f"Midfielder Shapiro-Wilk p-value: {mid_shapiro.pvalue:.4f}")


Forward Shapiro-Wilk p-value: 0.0000
Midfielder Shapiro-Wilk p-value: 0.0000


In [3]:
# Welch independent two-sample t-test, one-sided alternative.
#Perform a Welch's t-test to compare the means of the "SoT/90" values between the "Forward" and "Midfielder" position groups. The test is one-sided, testing the alternative hypothesis that the mean of the "Forward" group is greater than that of the "Midfielder" group. The `equal_var=False` parameter indicates that we do not assume equal variances between the two groups.
t_stat, p_value = st.ttest_ind( # Perform the t-test
    sample_fw,
    sample_mid,
    equal_var=False,
    alternative="greater"
)

alpha = 0.05  # Set the significance level (alpha) for the hypothesis test. This value represents the threshold for rejecting the null hypothesis, with a common choice being 0.05 (5%).

#report the results of the Welch two-sample t-test, including the means of both position groups, the mean difference, the t-statistic, the one-sided p-value, and the significance level (alpha). Based on the p-value and alpha, a conclusion is drawn regarding whether to reject or fail to reject the null hypothesis (H0).
print("=== WELCH TWO-SAMPLE T-TEST ===")
print(f"Forward sample mean:    {sample_fw.mean():.4f}")
print(f"Midfielder sample mean: {sample_mid.mean():.4f}")
print(f"Mean difference:        {sample_fw.mean() - sample_mid.mean():.4f}")
print(f"t-statistic:            {t_stat:.4f}")
print(f"One-sided p-value:      {p_value:.4f}")
print(f"Alpha:                  {alpha:.2f}")

print("\nStatistical conclusion:")
if p_value <= alpha:  # If the p-value is less than or equal to the significance level (alpha), reject the null hypothesis (H0) and conclude that there is sufficient evidence to support the alternative hypothesis that Forwards have a higher mean SoT/90 than Midfielders.
    print("Reject H0.")
    print("There is sufficient evidence that Forwards have a higher mean SoT/90 than Midfielders.")
else:
    print("Fail to reject H0.")
    print("There is insufficient evidence that Forwards have a higher mean SoT/90 than Midfielders.")


=== WELCH TWO-SAMPLE T-TEST ===
Forward sample mean:    0.6647
Midfielder sample mean: 0.5789
Mean difference:        0.0858
t-statistic:            0.5469
One-sided p-value:      0.2927
Alpha:                  0.05

Statistical conclusion:
Fail to reject H0.
There is insufficient evidence that Forwards have a higher mean SoT/90 than Midfielders.


In [4]:
# Export a small results table for reporting.
test_results = pd.DataFrame({   #create a pandas DataFrame named test_results to store the results of the Welch two-sample t-test, including the means of both position groups, the mean difference, the t-statistic, the one-sided p-value, the significance level (alpha), and the decision regarding the null hypothesis.
    "Forward_Mean": [sample_fw.mean()],   
    "Midfielder_Mean": [sample_mid.mean()],
    "Mean_Difference": [sample_fw.mean() - sample_mid.mean()],
    "t_statistic": [t_stat],
    "one_sided_p_value": [p_value],
    "alpha": [alpha],
    "decision": ["Reject H0" if p_value <= alpha else "Fail to reject H0"]
})

display(test_results.round(4)) # Display the results rounded to 4 decimal places


,Forward_Mean,Midfielder_Mean,Mean_Difference,t_statistic,one_sided_p_value,alpha,decision
0,0.6647,0.5789,0.0858,0.5469,0.2927,0.05,Fail to reject H0
